<font color= 'darkblue'>
    
# 📺YouTube Ad Revenue Prediction - Monetization Modeler

## Project Overview

This project develops a machine learning regression solution to predict
YouTube advertising revenue (`ad_revenue_usd`) using video performance,
engagement, audience, device, country, and content-related features.

The project covers:

- Data understanding and cleaning
- Exploratory Data Analysis (EDA)
- Feature engineering
- Categorical encoding
- Regression model development
- Model evaluation and comparison
- Feature interpretation
- Business insights
- Final model deployment through Streamlit

### Importing nessary libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import joblib

### Load Dataset

In [ ]:
df=pd.read_csv("youtube_ad_revenue_dataset (1).csv")

### Data Understanding

The dataset is inspected using shape, data types, descriptive statistics,
and sample records to understand its structure and variables.

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df['ad_revenue_usd'].value_counts()

### Data Cleaning

#### Dealing with duplicates

In [ ]:
dupes=df.duplicated()
sum(dupes)

#### drop duplicates


In [ ]:
df=df.drop_duplicates()

In [ ]:
 df.duplicated().sum()

### Correlation among pairs of continuous variables

In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df.corr(numeric_only=True), annot=True, linewidths=.5, fmt= '.1f', center = 1 )  # heatmap
plt.show()

<font color = 'darkblue'>

* from the above heatmap we can see watch_time_minutes and ad_revenue_usd is the only pair with most correlation

### Handling missing values

### Standard missing values

In [ ]:
df.isnull().sum()

In [ ]:
df1=pd.DataFrame({'value' : df['likes'], 'Missing?' : df['likes'].isnull()}) 
df1.head(50)

In [ ]:
df.dropna(inplace=True)           # drop the missing values
df.isnull().sum()

In [ ]:
df.isnull().any()

In [ ]:
df.value_counts().sum()

### Handling outliers 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = [
    "likes",
    "views",
    "comments",
    "watch_time_minutes",
    "video_length_minutes",
    "subscribers",
    "ad_revenue_usd"
]

fig, axes = plt.subplots(
    nrows=len(numeric_cols),
    ncols=2,
    figsize=(12, 28)
)

for i, col in enumerate(numeric_cols):

    # Distribution plot
    sns.histplot(
        data=df,
        x=col,
        kde=True,
        ax=axes[i, 0]
    )

    axes[i, 0].set_title(
        f"{col} Distribution",
        fontsize=10
    )

    # Boxplot
    sns.boxplot(
        data=df,
        x=col,
        ax=axes[i, 1]
    )

    axes[i, 1].set_title(
        f"{col} Boxplot",
        fontsize=10
    )

plt.tight_layout()
plt.show()

<font color= 'darkblue'>

from the above boxplot, views column only getting outliers.

In [ ]:
Q1 = df["views"].quantile(0.25)
Q3 = df["views"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

outliers = df[
    (df["views"] < lower_bound) |
    (df["views"] > upper_bound)
]

print("Number of outliers:", len(outliers))
print("Percentage:", len(outliers) / len(df) * 100)

In [ ]:
print("Views skewness:", df["views"].skew())

In [ ]:
df["views"].describe()

In [ ]:
print("Minimum:", df["views"].min())
print("Maximum:", df["views"].max())

<font color = 'darkblue'>

Outlier Analysis – Views

* The IQR method identified 865 observations (0.72%) as potential outliers in the views variable.
    
* However, the views distribution is approximately symmetric, with a skewness close to zero (-0.0029). The observed values range from 9,521 to 10,468.

* Since these observations fall within a reasonable range and do not indicate extreme or unrealistic values, they were retained rather than removed.

### Feature Engineering

#### Engagement Rate

<font color= 'darkblue'>

An engagement rate feature is created from likes, comments, and views
to represent audience interaction relative to video views.

In [ ]:
df["engagement_rate"] = (df["likes"] + df["comments"]) / df["views"]      # new column

### Encoding categorical variable

### One Hot Encoding

#### Categorical Encoding

<font color= 'darkblue'>

The categorical variables `category`, `device`, and `country` are
converted into numerical dummy variables using One-Hot Encoding.

`drop_first=True` is used to avoid redundant dummy variables and
reduce multicollinearity.

Reference categories:

- Education — Category
- Desktop — Device
- AU — Country

In [ ]:
df.info()

In [ ]:
df['category'].value_counts()

In [ ]:
df2 = pd.get_dummies(
    df,
    columns=["category", "device", "country"],
     dtype= int ,drop_first=True
)

In [ ]:
df2.head()

In [ ]:
df2.info()

### EDA

####  Revenue Distribution 

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["ad_revenue_usd"], bins=50)
plt.title("Revenue Distribution")
plt.show()

#### Correlation heatmap

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df2.corr(numeric_only=True), annot=True, linewidths=0.5, fmt= '.1f', center = 1 )
plt.title("Correlation Heatmap")
plt.show()

<font color= 'darkblue'>

Engagement rate improves monetization

#### Views vs Revenue

In [ ]:
sns.scatterplot(x=df["views"], y=df["ad_revenue_usd"])
plt.show()

<font color= 'darkblue'>

Views strongly influence ad revenue

#### Category-wise Revenue

In [ ]:
sns.boxplot(x=df["category"], y=df["ad_revenue_usd"])
plt.show()

### DEFINE FEATURES & TARGET

In [ ]:
y = df2["ad_revenue_usd"]              # target variable

In [ ]:
X = df2.drop(columns=["ad_revenue_usd", "video_id", "date"])

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split (X, y,test_size=0.3, random_state=100)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
X_test.head()

In [ ]:
y_train.shape

In [ ]:
y_test.shape

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()

In [ ]:
lr.fit(X_train,y_train)

In [ ]:
# The coefficients
print('Coefficients: \n', lr.coef_)

### Predict the model

In [ ]:
lr_pred_train = lr.predict( X_train)

In [ ]:
lr_pred_train

In [ ]:
lr_pred_test = lr.predict( X_test)

In [ ]:
lr_pred_test


In [ ]:
from sklearn import metrics

In [ ]:
train_score=metrics.r2_score(y_train,lr_pred_train)
train_score

In [ ]:
test_score=metrics.r2_score(y_test,lr_pred_test)
linear_r2=test_score
linear_r2

### Evaluate the model

In [ ]:
from sklearn import metrics

linear_mae= metrics.mean_absolute_error(y_test, lr_pred_test)
linear_mse= metrics.mean_squared_error(y_test, lr_pred_test)
linear_rmse= np.sqrt(metrics.mean_squared_error(y_test, lr_pred_test))

print("Linear Regression")
print("R²:", linear_r2)
print("MAE:", linear_mae)
print("MSE:", linear_mse)
print("RMSE:", linear_rmse)

### Ridge Regression

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print(X_train_scaled[:5])

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge=Ridge()
parameters={'alpha':[0.1,1,5,10,20,30,35,40,45,50,55,100]}## Appllying Grid Search
ridge_regressor=GridSearchCV(ridge,parameters,scoring='neg_mean_squared_error',cv=5)
ridge_regressor.fit(X_train_scaled,y_train)

In [ ]:
print(ridge_regressor.best_params_)## Best value for the Parameter

In [ ]:
from sklearn.linear_model import Ridge## Building the Model with the best parameter value
ridge_regressor = Ridge(alpha = 0.1)
ridge_regressor.fit(X_train_scaled, y_train)

In [ ]:
ridge_pred_train=ridge_regressor.predict(X_train_scaled)

In [ ]:
ridge_pred_test=ridge_regressor.predict(X_test_scaled)

In [ ]:
# R square on training data
train_score=metrics.r2_score(y_train,ridge_pred_train)
train_score

In [ ]:
test_score=metrics.r2_score(y_test,ridge_pred_test)
ridge_r2 = test_score
ridge_r2

In [ ]:
lr.coef_

In [ ]:
ridge_regressor.coef_

In [ ]:
# Let us explore the coefficients for each of the independent attributes

for idx, col_name in enumerate(X_train.columns):
    print(f"The coefficient for {col_name} is {ridge_regressor.coef_[idx]}")

In [ ]:

ridge_mae= mean_absolute_error(y_test, ridge_pred_test)
ridge_mse= mean_squared_error(y_test, ridge_pred_test)
ridge_rmse= np.sqrt(mean_squared_error(y_test, ridge_pred_test))

print("Ridge Regression")
print("R²:", ridge_r2)
print("MAE:", ridge_mae)
print("MSE:", ridge_mse)
print("RMSE:", ridge_rmse)

### Laso Regression

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
lasso=Lasso()
parameters={'alpha':[0.1,1,5,10,20,30,35,40,45,50,55,100]}
lasso_regressor=GridSearchCV(lasso,parameters,scoring='neg_mean_squared_error',cv=5)

lasso_regressor.fit(X_train_scaled,y_train)

In [ ]:
lasso_regressor.best_params_

In [ ]:
from sklearn.linear_model import Lasso
lasso_regressor = Lasso(alpha =  0.1 )
lasso_regressor.fit(X_train_scaled,y_train)

In [ ]:
lasso_pred_train=lasso_regressor.predict(X_train_scaled)

In [ ]:
lasso_pred_test=lasso_regressor.predict(X_test_scaled)

In [ ]:
# R square on training data
train_score=metrics.r2_score(y_train,lasso_pred_train)
train_score

In [ ]:
# R square on training data
test_score=metrics.r2_score(y_test,lasso_pred_test)
lasso_r2 = test_score
lasso_r2

In [ ]:
lasso_regressor.coef_

In [ ]:

lasso_mae= mean_absolute_error(y_test, lasso_pred_test)
lasso_mse= mean_squared_error(y_test, lasso_pred_test)
lasso_rmse= np.sqrt(mean_squared_error(y_test, lasso_pred_test))

print("Lasso Regression")
print("R²:", lasso_r2)
print("MAE:", lasso_mae)
print("MSE:", lasso_mse)
print("RMSE:", lasso_rmse)

### Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=30,
    n_jobs=-1
)

rf_regressor.fit(X_train, y_train)



In [ ]:
rf_pred_test = rf_regressor.predict(X_test)

In [ ]:
rf_pred_train = rf_regressor.predict(X_train)

In [ ]:
# R square on training data
train_score=metrics.r2_score(y_train,rf_pred_train)
train_score

In [ ]:
# R square on testing data
test_score=metrics.r2_score(y_test,rf_pred_test)
rf_r2 = test_score
rf_r2

In [ ]:
from sklearn import metrics


rf_mae= metrics.mean_absolute_error(y_test, rf_pred_test)
rf_mse= metrics.mean_squared_error(y_test, rf_pred_test)
rf_rmse= np.sqrt(metrics.mean_squared_error(y_test, rf_pred_test))

print("Random Forest Regression")
print("R²:", rf_r2)
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)

### Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_regressor = GradientBoostingRegressor(
    random_state=42
)

gb_regressor.fit(X_train, y_train)

In [ ]:
gb_pred_test = gb_regressor.predict(X_test)

In [ ]:
gb_pred_train = gb_regressor.predict(X_train)

In [ ]:
# R square on training data
train_score=metrics.r2_score(y_train,gb_pred_train)
train_score

In [ ]:
# R square on testing data
test_score=metrics.r2_score(y_test,gb_pred_test)
gb_r2 = test_score
gb_r2

In [ ]:
from sklearn import metrics

gb_mae= metrics.mean_absolute_error(y_test, gb_pred_test)
gb_mse= metrics.mean_squared_error(y_test, gb_pred_test)
gb_rmse= np.sqrt(metrics.mean_squared_error(y_test, gb_pred_test))

print("Gradient Boosting")
print("R²:", gb_r2)
print("MAE:", gb_mae)
print("MSE:", gb_mse)
print("RMSE:", gb_rmse)

### Final Result

In [ ]:
results_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "Random Forest",
        "Gradient Boosting"
    ],

    "R²": [
        linear_r2,
        ridge_r2,
        lasso_r2,
        rf_r2,
        gb_r2
    ],

    "MAE": [
        linear_mae,
        ridge_mae,
        lasso_mae,
        rf_mae,
        gb_mae
    ],

    "MSE": [
        linear_mse,
        ridge_mse,
        lasso_mse,
        rf_mse,
        gb_mse
    ],

    "RMSE": [
        linear_rmse,
        ridge_rmse,
        lasso_rmse,
        rf_rmse,
        gb_rmse
    ]
})

results_df

<font color = 'darkblue'>

Model Comparison and Selection

* Five regression models—Linear Regression, Ridge Regression, Lasso Regression, Random Forest Regression, and Gradient Boosting Regression—were trained to predict ad_revenue_usd. The models were evaluated using R², MAE, MSE, and RMSE.

* Linear Regression achieved the best overall performance, with an R² score of 1.000000 and extremely low MAE and RMSE values. Ridge Regression also performed exceptionally well, while Lasso Regression, Random Forest, and Gradient Boosting showed slightly higher prediction errors.

* Based on the evaluation metrics, <font color = 'green'>Linear Regression <font color = 'darkblue'>was selected as the final model for the application.

### Conclusion

<font color= 'green'>

Linear Regression was selected as the final model because it achieved the strongest overall performance, with an R² of approximately 1.00 and extremely low error metrics.

### Feature Interpretation

In [ ]:
# Get feature names
feature_names = X_train.columns

# Get coefficients
coefficients = lr.coef_

# Create a dataframe
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})


coef_df

In [ ]:
plt.figure(figsize=(10, 8))

sns.barplot(
    data=coef_df,
    x="Coefficient",
    y="Feature"
)

plt.title("Linear Regression Feature Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
corr_with_revenue = (
    df.select_dtypes(include="number")
      .corr()["ad_revenue_usd"]
      .sort_values(ascending=False)
)

corr_with_revenue

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_regressor.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "Importance",
    ascending=False
)

rf_importance.head(10)

<font color = 'darkblue'>

* Feature interpretation was performed using Linear Regression coefficients and Random Forest feature importance. Linear Regression showed positive coefficients for comments, likes, views, and watch_time_minutes, indicating positive modeled relationships with ad revenue.

* Random Forest feature importance identified watch_time_minutes as the dominant predictive feature, with an importance of approximately 0.978, followed by engagement_rate with approximately 0.0215. The remaining variables had relatively small importance values.

* The strong importance of watch_time_minutes is consistent with the exploratory correlation analysis, where watch_time_minutes had a correlation of approximately 0.989 with ad_revenue_usd. Therefore, <font color = 'green'>watch time appears to be the strongest predictor of ad revenue in this dataset.

### Business Insights

#### Key Findings

<font color= 'darkblue'>

1. **Watch time is the strongest predictor**
   
   Random Forest feature importance assigned approximately 97.8% of its
   importance score to `watch_time_minutes`.

2. **Watch time has a very strong relationship with revenue**
   
   The correlation between `watch_time_minutes` and `ad_revenue_usd`
   is approximately 0.989.

3. **Engagement-related variables are useful**
   
   Likes, comments, views, and engagement rate provide additional
   information about predicted revenue.

4. **Categorical variables have relatively small predictive importance**
   
   Category, device, and country contributed much less to the final
   tree-based feature importance compared with watch time.

5. **The dataset is synthetic**
   
   Therefore, these relationships should be interpreted as model-based
   associations rather than causal conclusions.

### Save Final Model

In [ ]:
import joblib

# Save the exact feature names used by the model
feature_names = X.columns.tolist()

# Save model + feature names together
model_bundle = {
    "model": lr,
    "feature_names": feature_names
}

joblib.dump(model_bundle, "linear_regression_model.pkl")

print("Model saved successfully!")

### Model Verification

In [ ]:
import os

print(os.path.exists("linear_regression_model.pkl"))

In [ ]:
print(os.path.getsize("linear_regression_model.pkl"), "bytes")

In [ ]:
loaded_bundle = joblib.load("linear_regression_model.pkl")

loaded_model = loaded_bundle["model"]
loaded_features = loaded_bundle["feature_names"]

print(type(loaded_model))
print(len(loaded_features))
print(loaded_features)

In [ ]:
loaded_predictions = loaded_model.predict(X_test)

print("First 10 predictions:")
print(loaded_predictions[:10])

In [ ]:
original_predictions = lr.predict(X_test)

print(
    "Predictions identical:",
    np.allclose(
        original_predictions,
        loaded_predictions
    )
)

In [ ]:
print("Categories:")
print(df["category"].unique())

print("\nDevices:")
print(df["device"].unique())

print("\nCountries:")
print(df["country"].unique())

print("\nModel features:")
print(X.columns.tolist())